# KOSIS 임베딩 인덱스 구축 (Colab GPU)

뉴스 주장 → KOSIS 통계표 검색용 임베딩 인덱스를 만든다. (107,138개 표)

- **모델**: BGE-M3 (기본) / KURE-v1 (한국어 특화, A/B용)
- **결과**: `data/indexes/kosis_bge_m3/` — 로컬로 내려서 `--retrieval-mode hybrid` 실행에 사용
- **주의**: 반드시 GPU 런타임. 인덱스 구축까지만 Colab에서 하고, hybrid 검증은 로컬(.env 필요)에서.

## 1단계: GPU 런타임 확인
상단 메뉴 → 런타임 → 런타임 유형 변경 → 하드웨어 가속기 **GPU** 선택 후 아래 실행.

In [ ]:
!nvidia-smi -L  # GPU 잡혔는지 확인 (안 나오면 런타임 유형을 GPU로 바꾸세요)

## 2단계: 저장소 클론 + 의존성 설치

In [ ]:
!git clone https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git
%cd NLP_05-Team-Project-3
!git checkout Poc
!pip install -q -r requirements-ml.txt

In [ ]:
# 표 인덱스 파일 확인 (107,138행이어야 함)
import pandas as pd
df = pd.read_csv('kosis_table_summary.csv')
print('통계표 수:', len(df))
df.head(2)

## 3단계: BGE-M3 임베딩 인덱스 구축
107k개 표를 벡터로 인코딩한다. GPU로 수~수십 분. `saved=... tables=107138 dimension=1024`가 뜨면 성공.

In [ ]:
!python kosis_build_embedding_index.py \
    --table-index kosis_table_summary.csv \
    --out-dir data/indexes/kosis_bge_m3 \
    --device cuda

## 4단계 (선택): KURE-v1 인덱스도 구축 — A/B 비교용
KURE는 BGE-M3를 한국어로 파인튜닝한 상위호환. 모델명·출력경로만 다르다. 시간 없으면 건너뛰고 3단계 것만 써도 됨.

In [ ]:
!python kosis_build_embedding_index.py \
    --table-index kosis_table_summary.csv \
    --embedding-model nlpai-lab/KURE-v1 \
    --out-dir data/indexes/kosis_kure \
    --device cuda

## 5단계: 인덱스 내보내기
### 방법 A — 바로 다운로드 (zip)

In [ ]:
!zip -r kosis_indexes.zip data/indexes
from google.colab import files
files.download('kosis_indexes.zip')  # 로컬로 다운로드 → 압축 풀어 data/indexes/ 에 배치

### 방법 B — Google Drive에 저장 (팀 공유·재사용 편함, 인덱스가 400MB대라 추천)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/kosis_indexes
!cp -r data/indexes/* /content/drive/MyDrive/kosis_indexes/
print('드라이브에 저장 완료: MyDrive/kosis_indexes/')

## 6단계: 로컬에서 할 일 (Colab 아님)

내려받은 인덱스를 로컬 저장소의 `data/indexes/`에 배치한 뒤, 로컬에서 hybrid로 재실행·채점한다 (KOSIS API `.env` 필요).

```bash
# BGE-M3 hybrid
python run_kosis_measurement_pipeline.py --input hcx_v15_monthfix.csv --table-index kosis_table_summary.csv --retrieval-mode hybrid --verify
python score_gold.py --gold-measurement gold_measurement_final.csv --candidates outputs/runs/hcx_v15_monthfix_kosis_table_candidates.csv --verified outputs/runs/hcx_v15_monthfix_kosis_verified.csv

# KURE 비교 (인덱스 경로만 교체)
python run_kosis_measurement_pipeline.py --input hcx_v15_monthfix.csv --table-index kosis_table_summary.csv --retrieval-mode hybrid --semantic-index data/indexes/kosis_kure --verify
```

**볼 것**: `recall@5`가 lexical의 62.5%에서 얼마나 오르는지. 그게 임베딩 도입 효과이며, 판정 정확도 천장(현재 57%)을 끌어올린다.